# 🧠 Financial Product Recommendation System with Deep Learning & UltraGCN
## Huấn luyện & So sánh 5 Thuật toán Gợi ý Sản phẩm Ngân hàng & Tài chính Cá nhân

Notebook này triển khai và so sánh toàn diện 5 mô hình recommendation trên tập dữ liệu tương tác khách hàng - sản phẩm tài chính:
1. **Matrix Factorization (MF)** (Collaborative Filtering phân rã ma trận ẩn).
2. **Neural Collaborative Filtering (NCF)** (Học phi tuyến qua mạng neural).
3. **Deep Multi-Layer Perceptron (Deep MLP)** (Mạng truyền thẳng sâu).
4. **LightGCN** (Graph Convolutional Network trên đồ thị tương tác).
5. **Financial-UltraGCN** (Mô hình đồ thị tối ưu hóa trực tiếp bỏ qua message passing đắt đỏ).

**Đánh giá trên 5 chỉ số tiêu chuẩn:** `Precision@10`, `Recall@10`, `NDCG@10`, `RMSE`, `MAE`.

### 1. Import Thư viện & Cấu hình Môi trường

In [ ]:
import os
import sys
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from collections import defaultdict

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'⚡ Thiết bị tính toán: {device}')

### 2. Nạp Dữ liệu Tương tác & Chuẩn bị Tập Train / Test

In [ ]:
DATA_DIR = 'data_fintech' if os.path.exists('data_fintech') else 'E:/Fintech/Fintech'

df_customers = pd.read_csv(os.path.join(DATA_DIR, 'dim_customer.csv'))
df_products = pd.read_csv(os.path.join(DATA_DIR, '03_danh_muc_san_pham.csv'))
df_transactions = pd.read_csv(os.path.join(DATA_DIR, 'fact_transaction.csv'))
df_holdings = pd.read_csv(os.path.join(DATA_DIR, 'fact_customer_product.csv'))

service_to_prod = {
    'A. Tài khoản & Thông tin KH': 'SP024',
    'B. Giao dịch tiền mặt': 'SP004',
    'C. Tiết kiệm & Tiền gửi': 'SP001',
    'D. Thẻ': 'SP004',
    'E. Chuyển tiền & Thanh toán': 'SP021',
    'F. Ngoại tệ': 'SP018',
    'G. Tín dụng': 'SP008',
    'H. Bảo hiểm & Đầu tư': 'SP013',
    'I. Ngân hàng số & Hỗ trợ': 'SP021',
    'J. Khách hàng doanh nghiệp': 'SP025'
}

interactions = []
for _, row in df_holdings.iterrows():
    c_id = int(row['customer_id'])
    p_id = str(row['product_id'])
    bal = float(row.get('current_balance', 10_000_000))
    rating = 4.5 + min(0.5, np.log10(max(1, bal)) / 10.0)
    interactions.append({'customer_id': c_id, 'product_id': p_id, 'rating': rating})

for _, row in df_transactions.iterrows():
    c_id = int(row['customer_id'])
    s_grp = str(row['service_group'])
    p_id = service_to_prod.get(s_grp, 'SP001')
    amt = float(row.get('amount', 1_000_000))
    rating = min(4.5, 1.0 + (np.log1p(amt) / 18.0) * 3.5)
    interactions.append({'customer_id': c_id, 'product_id': p_id, 'rating': rating})

df_inter = pd.DataFrame(interactions)
df_grouped = df_inter.groupby(['customer_id', 'product_id'])['rating'].mean().reset_index()

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
df_grouped['user_id'] = user_encoder.fit_transform(df_grouped['customer_id'])
df_grouped['item_id'] = item_encoder.fit_transform(df_grouped['product_id'])

num_users = len(user_encoder.classes_)
num_items = len(item_encoder.classes_)
num_nodes = num_users + num_items

train_df, test_df = train_test_split(df_grouped, test_size=0.2, random_state=SEED)

train_u = torch.tensor(train_df['user_id'].values, dtype=torch.long)
train_i = torch.tensor(train_df['item_id'].values, dtype=torch.long)
train_r = torch.tensor(train_df['rating'].values, dtype=torch.float32)

test_u = torch.tensor(test_df['user_id'].values, dtype=torch.long)
test_i = torch.tensor(test_df['item_id'].values, dtype=torch.long)
test_r = torch.tensor(test_df['rating'].values, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(train_u, train_i, train_r), batch_size=256, shuffle=True)

# Đồ thị Bipartite
user_freq = defaultdict(int)
item_freq = defaultdict(int)
for row in train_df.itertuples():
    user_freq[row.user_id] += 1
    item_freq[row.item_id] += 1

edge_index, edge_weight = [], []
for row in train_df.itertuples():
    u, i = row.user_id, row.item_id
    edge_index.append([u, num_users + i])
    edge_index.append([num_users + i, u])
    w = 1.0 / ((max(1, user_freq[u]) ** 0.5) * (max(1, item_freq[i]) ** 0.5))
    edge_weight.append(w)
    edge_weight.append(w)

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight, dtype=torch.float32)
print(f'✅ Tập huấn luyện: {len(train_df)} | Tập kiểm thử: {len(test_df)} | Số node đồ thị: {num_nodes}')

### 3. Định nghĩa Kiến trúc 5 Mô hình Recommendation

In [ ]:
EMB_DIM = 32

# 1. Matrix Factorization
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=EMB_DIM):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    def forward(self, user, item):
        return (self.user_emb(user) * self.item_emb(item)).sum(dim=-1)

# 2. NCF
class NCF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=EMB_DIM):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        self.mlp = nn.Sequential(nn.Linear(emb_dim * 2, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, user, item):
        return self.mlp(torch.cat([self.user_emb(user), self.item_emb(item)], dim=-1)).squeeze(-1)

# 3. Deep MLP
class DeepMLP(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=EMB_DIM):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        self.net = nn.Sequential(nn.Linear(emb_dim * 2, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, user, item):
        return self.net(torch.cat([self.user_emb(user), self.item_emb(item)], dim=-1)).squeeze(-1)

# 4. LightGCN
class LightGCN(nn.Module):
    def __init__(self, num_nodes, emb_dim=EMB_DIM):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, emb_dim)
        nn.init.xavier_uniform_(self.emb.weight)
    def forward(self, edge_index):
        x = self.emb.weight
        row, col = edge_index
        deg = torch.bincount(row, minlength=x.size(0)).float().clamp(min=1)
        norm = 1.0 / deg[row].sqrt() * 1.0 / deg[col].sqrt()
        out = torch.zeros_like(x)
        out.index_add_(0, row, x[col] * norm.unsqueeze(1))
        return out

# 5. UltraGCN
class UltraGCN(nn.Module):
    def __init__(self, num_nodes, emb_dim=EMB_DIM):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, emb_dim)
        nn.init.xavier_uniform_(self.emb.weight)
    def forward(self, edge_index, edge_weight):
        x = self.emb.weight
        row, col = edge_index
        norm = edge_weight
        out = torch.zeros_like(x)
        out.index_add_(0, row, x[col] * norm.unsqueeze(1))
        return out

### 4. Huấn luyện 5 Mô hình & Đánh giá Hiệu năng So sánh

In [ ]:
EPOCHS = 15
criterion = nn.MSELoss()

# Khởi tạo mô hình
m_mf = MatrixFactorization(num_users, num_items)
m_ncf = NCF(num_users, num_items)
m_mlp = DeepMLP(num_users, num_items)
m_light = LightGCN(num_nodes)
m_ultra = UltraGCN(num_nodes)

opt_mf = optim.Adam(m_mf.parameters(), lr=0.005)
opt_ncf = optim.Adam(m_ncf.parameters(), lr=0.005)
opt_mlp = optim.Adam(m_mlp.parameters(), lr=0.005)
opt_light = optim.Adam(m_light.parameters(), lr=0.005)
opt_ultra = optim.Adam(m_ultra.parameters(), lr=0.005)

for epoch in range(EPOCHS):
    # Train MF, NCF, MLP
    for u, i, r in train_loader:
        for m, opt in [(m_mf, opt_mf), (m_ncf, opt_ncf), (m_mlp, opt_mlp)]:
            m.train()
            opt.zero_grad()
            loss = criterion(m(u, i), r)
            loss.backward()
            opt.step()
            
    # Train LightGCN
    m_light.train()
    opt_light.zero_grad()
    emb_l = m_light(edge_index)
    loss_l = criterion((emb_l[train_u] * emb_l[num_users + train_i]).sum(dim=-1), train_r) + 1e-4 * torch.norm(emb_l)
    loss_l.backward()
    opt_light.step()
    
    # Train UltraGCN
    m_ultra.train()
    opt_ultra.zero_grad()
    emb_u = m_ultra(edge_index, edge_weight)
    loss_u = criterion((emb_u[train_u] * emb_u[num_users + train_i]).sum(dim=-1), train_r) + 1e-4 * torch.norm(emb_u)
    loss_u.backward()
    opt_ultra.step()

print('✅ Đã huấn luyện xong toàn bộ 5 mô hình.')

### 5. Bảng So sánh Hiệu năng (Precision@10, Recall@10, NDCG@10, RMSE, MAE)

In [ ]:
def eval_model(name, model, is_graph=False, is_ultra=False):
    model.eval()
    with torch.no_grad():
        if not is_graph:
            preds = model(test_u, test_i).cpu().numpy()
            all_scores = (model.user_emb.weight.cpu().numpy() @ model.item_emb.weight.cpu().numpy().T)
        elif is_ultra:
            emb = model(edge_index, edge_weight).cpu().numpy()
            preds = (emb[test_u] * emb[num_users + test_i]).sum(axis=-1)
            all_scores = emb[:num_users] @ emb[num_users:].T
        else:
            emb = model(edge_index).cpu().numpy()
            preds = (emb[test_u] * emb[num_users + test_i]).sum(axis=-1)
            all_scores = emb[:num_users] @ emb[num_users:].T
            
        targets = test_r.cpu().numpy()
        rmse = float(np.sqrt(np.mean((preds - targets) ** 2)))
        mae = float(np.mean(np.abs(preds - targets)))
        
        user_items = defaultdict(set)
        for u, i, r in zip(test_u.numpy(), test_i.numpy(), targets):
            if r >= 3.0:
                user_items[u].add(i)
                
        ps, rs, ndcgs = [], [], []
        for u in user_items:
            true_items = user_items[u]
            top_k = np.argsort(all_scores[u])[-10:][::-1]
            hits = len(set(top_k) & true_items)
            ps.append(hits / 10.0)
            rs.append(hits / len(true_items))
            dcg = sum([1.0 / np.log2(idx + 2) for idx, itm in enumerate(top_k) if itm in true_items])
            idcg = sum([1.0 / np.log2(idx + 2) for idx in range(min(10, len(true_items)))])
            ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
            
        return {
            'Model': name,
            'Precision@10': round(float(np.mean(ps)), 4),
            'Recall@10': round(float(np.mean(rs)), 4),
            'NDCG@10': round(float(np.mean(ndcgs)), 4),
            'RMSE': round(rmse, 4),
            'MAE': round(mae, 4)
        }

res = [
    eval_model('Matrix Factorization (MF)', m_mf),
    eval_model('Neural Collaborative Filtering (NCF)', m_ncf),
    eval_model('Deep Multi-Layer Perceptron (Deep MLP)', m_mlp),
    eval_model('LightGCN', m_light, is_graph=True),
    eval_model('Financial-UltraGCN', m_ultra, is_graph=True, is_ultra=True)
]
df_res = pd.DataFrame(res)
df_res

### 6. Trực quan hóa Biểu đồ So sánh Hiệu năng

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
df_rank = df_res.melt(id_vars=['Model'], value_vars=['Precision@10', 'Recall@10', 'NDCG@10'], var_name='Metric', value_name='Score')
sns.barplot(data=df_rank, x='Model', y='Score', hue='Metric', palette='viridis')
plt.title('So sánh Ranking Metrics (Precision@10, Recall@10, NDCG@10)', fontweight='bold')
plt.xticks(rotation=25, ha='right')
plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
df_err = df_res.melt(id_vars=['Model'], value_vars=['RMSE', 'MAE'], var_name='Metric', value_name='Error')
sns.barplot(data=df_err, x='Model', y='Error', hue='Metric', palette='magma')
plt.title('So sánh Error Metrics (RMSE & MAE - Càng thấp càng tốt)', fontweight='bold')
plt.xticks(rotation=25, ha='right')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()